In [3]:
# !pip install torch torch-geometric 
# !pip install -U scikit-learn

In [4]:
# import torch

# TORCH = torch.__version__.split('+')[0]
# CUDA = "cu" + torch.version.cuda.replace('.', '') if torch.cuda.is_available() else "cpu"

# !pip install torch-scatter torch-sparse pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

In [5]:
import torch
import torch_geometric as tg
from torch_geometric.datasets import Yelp

from torch_geometric.loader import NeighborLoader

from torch_geometric.loader import GraphSAINTNodeSampler
from torch_geometric.nn import GraphSAGE

import itertools
import numpy as np
import time

from sklearn.metrics import f1_score
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support

/home/soham/miniconda3/envs/GNNProject/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
dataset = Yelp("./data/Yelp")

In [7]:
dataset

Yelp()

In [8]:
data = dataset[0]
data

Data(x=[716847, 300], edge_index=[2, 13954819], y=[716847, 100], train_mask=[716847], val_mask=[716847], test_mask=[716847])

In [9]:
dataset.num_node_features

300

In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [11]:
train_loader_NodeSampler = GraphSAINTNodeSampler(data, batch_size=6000, num_steps=40, save_dir='./subgraphs_nodesampler')
print("Loader initialized successfully!")

Loader initialized successfully!


In [12]:
test_loader_NodeSampler = GraphSAINTNodeSampler(data, batch_size=6000, num_steps=60, save_dir='./subgraphs_nodesampler')
print("Loader initialized successfully!")

Loader initialized successfully!


In [13]:
train_loader = NeighborLoader(
    data,
    num_neighbors=[25, 10], 
    batch_size=1024,
    input_nodes=data.train_mask,
    shuffle=True
)

In [14]:
test_loader = NeighborLoader(
    data,
    num_neighbors=[-1], # -1 means "sample all neighbors" (use with caution on Yelp)
    batch_size=1024,
    input_nodes=data.test_mask,
    shuffle=False
)

In [15]:
temp_train = next(iter(train_loader_NodeSampler))
# temp_train = next(iter(train_loader))

In [16]:
temp_train

Data(num_nodes=5730, edge_index=[2, 180048], x=[5730, 300], y=[5730, 100], train_mask=[5730], val_mask=[5730], test_mask=[5730])

In [17]:
# temp_train.edge_index[0, :] == temp_train.edge_index[1, :]

In [18]:
# temp_train.edge_index.shape
counter = list(temp_train.edge_index[0, :] == temp_train.edge_index[1, :]).count(True)

# just confirming that the number of self-nodes is equal to the number of nodes in the batch
print(counter, temp_train.num_nodes)

5730 5730


In [19]:
hidden_channels_h = [128, 256, 512]
num_layers_h = [1, 2, 3, 4, 6, 8]
train_sampler_batch_sizes_h = [6000, 8000]
train_sampler_num_steps_h = [40, 60]
test_sampler_batch_sizes_h = [6000, 8000]
test_sampler_num_steps_h = [60, 80]

In [23]:
for item in list(itertools.product(hidden_channels_h, num_layers_h, train_sampler_batch_sizes_h, train_sampler_num_steps_h, test_sampler_batch_sizes_h, test_sampler_num_steps_h)):
    print(item)


(128, 1, 6000, 40, 6000, 60)
(128, 1, 6000, 40, 6000, 80)
(128, 1, 6000, 40, 8000, 60)
(128, 1, 6000, 40, 8000, 80)
(128, 1, 6000, 60, 6000, 60)
(128, 1, 6000, 60, 6000, 80)
(128, 1, 6000, 60, 8000, 60)
(128, 1, 6000, 60, 8000, 80)
(128, 1, 8000, 40, 6000, 60)
(128, 1, 8000, 40, 6000, 80)
(128, 1, 8000, 40, 8000, 60)
(128, 1, 8000, 40, 8000, 80)
(128, 1, 8000, 60, 6000, 60)
(128, 1, 8000, 60, 6000, 80)
(128, 1, 8000, 60, 8000, 60)
(128, 1, 8000, 60, 8000, 80)
(128, 2, 6000, 40, 6000, 60)
(128, 2, 6000, 40, 6000, 80)
(128, 2, 6000, 40, 8000, 60)
(128, 2, 6000, 40, 8000, 80)
(128, 2, 6000, 60, 6000, 60)
(128, 2, 6000, 60, 6000, 80)
(128, 2, 6000, 60, 8000, 60)
(128, 2, 6000, 60, 8000, 80)
(128, 2, 8000, 40, 6000, 60)
(128, 2, 8000, 40, 6000, 80)
(128, 2, 8000, 40, 8000, 60)
(128, 2, 8000, 40, 8000, 80)
(128, 2, 8000, 60, 6000, 60)
(128, 2, 8000, 60, 6000, 80)
(128, 2, 8000, 60, 8000, 60)
(128, 2, 8000, 60, 8000, 80)
(128, 3, 6000, 40, 6000, 60)
(128, 3, 6000, 40, 6000, 80)
(128, 3, 6000,

In [24]:
model = GraphSAGE(
    in_channels=dataset.num_node_features, # 300 
    hidden_channels=256,
    num_layers=2,
    out_channels=dataset.num_classes,      # Multi-label output 
).to(device)

In [25]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()

In [26]:
def train(model, loader):
    model.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # The built-in GraphSAGE takes (x, edge_index)
        out = model(batch.x, batch.edge_index)
        
        # Calculate loss on training nodes within the sampled subgraph [cite: 19]
        loss = criterion(out[batch.train_mask], batch.y[batch.train_mask])
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [27]:
def test(model, loader):
    model.eval()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        
        # The built-in GraphSAGE takes (x, edge_index)
        out = model(batch.x, batch.edge_index)
        
        # Calculate loss on training nodes within the sampled subgraph [cite: 19]
        loss = criterion(out[batch.train_mask], batch.y[batch.train_mask])
        total_loss += loss.item()
    return total_loss / len(loader)

In [28]:
for epoch in range(200):
    loss = train(train_loader_NodeSampler)
    print(f'Epoch: {epoch+1:02d}, Loss: {loss:.4f}')

TypeError: train() missing 1 required positional argument: 'loader'

In [29]:
test_loss = test(test_loader_NodeSampler)
print(f'Loss: {loss:.4f}')

TypeError: test() missing 1 required positional argument: 'loader'

# Main run for all the combinations and results save

## hidden_channels_h = [128, 256, 512]
## num_layers_h = [1, 2, 3]
## train_sampler_batch_sizes_h = [4000, 6000, 8000, 10_000]
## train_sampler_num_steps_h = [40, 60, 80]
## test_sampler_batch_sizes_h = [4000, 6000, 8000, 10_000]
## test_sampler_num_steps_h = [40, 60, 80]

In [33]:
already_done = ["128 1 6000 40 6000 60",
"128 2 6000 40 6000 60",
"128 3 6000 40 6000 60",
"256 1 6000 40 6000 60",
"256 2 6000 40 6000 60",
"256 3 6000 40 6000 60",
"512 1 6000 40 6000 60",
"512 2 6000 40 6000 60",
"512 3 6000 40 6000 60",
"128 1 6000 40 6000 80",
"128 2 6000 40 6000 80",
"128 3 6000 40 6000 80",
"256 1 6000 40 6000 80",
"256 2 6000 40 6000 80",
"256 3 6000 40 6000 80",
"512 1 6000 40 6000 80",
"512 2 6000 40 6000 80",
"512 3 6000 40 6000 80",
"128 1 6000 40 8000 60",
"128 2 6000 40 8000 60",
"128 3 6000 40 8000 60",
"256 1 6000 40 8000 60",
"256 2 6000 40 8000 60",
"256 3 6000 40 8000 60",
"512 1 6000 40 8000 60",
"512 2 6000 40 8000 60",
"512 3 6000 40 8000 60",
"128 1 6000 40 8000 80",
"128 2 6000 40 8000 80",
"128 3 6000 40 8000 80",
"256 1 6000 40 8000 80",
"256 2 6000 40 8000 80",
"256 3 6000 40 8000 80",
"512 1 6000 40 8000 80",
"512 2 6000 40 8000 80",
"512 3 6000 40 8000 80",
"128 1 6000 60 6000 60",
"128 2 6000 60 6000 60",
"128 3 6000 60 6000 60",
"256 1 6000 60 6000 60",
"256 2 6000 60 6000 60",
"256 3 6000 60 6000 60",
"512 1 6000 60 6000 60",
"512 2 6000 60 6000 60",
"512 3 6000 60 6000 60",
"128 1 6000 60 6000 80",
"128 2 6000 60 6000 80",
"128 3 6000 60 6000 80",
"256 1 6000 60 6000 80",
"256 2 6000 60 6000 80",
"256 3 6000 60 6000 80",
"512 1 6000 60 6000 80",
"512 2 6000 60 6000 80",
"512 3 6000 60 6000 80",
"128 1 6000 60 8000 60",
"128 2 6000 60 8000 60",
"128 3 6000 60 8000 60",
"256 1 6000 60 8000 60",
"256 2 6000 60 8000 60",
"256 3 6000 60 8000 60",
"512 1 6000 60 8000 60",
"512 2 6000 60 8000 60",
"512 3 6000 60 8000 60",
"128 1 6000 60 8000 80",
"128 2 6000 60 8000 80",
"128 3 6000 60 8000 80",
"256 1 6000 60 8000 80",
"256 2 6000 60 8000 80",
"256 3 6000 60 8000 80",
"512 1 6000 60 8000 80",
"512 2 6000 60 8000 80",
"512 3 6000 60 8000 80",
"128 1 8000 40 6000 60",
"128 2 8000 40 6000 60"]

In [34]:
# np.save('./results/test.npy', np.array([0, 1, 2]))

In [ ]:
for c in train_sampler_batch_sizes_h:
    for d in train_sampler_num_steps_h:
        train_loader_main_run = GraphSAINTNodeSampler(data, batch_size=c, num_steps=d, save_dir=f'./subgraphs_nodesampler_{c}_{d}')

        for e in test_sampler_batch_sizes_h:
            for f in test_sampler_num_steps_h:
                test_loader_main_run = GraphSAINTNodeSampler(data, batch_size=e, num_steps=f, save_dir=f'./subgraphs_nodesampler_test_{e}_{f}')

                for a in hidden_channels_h:
                    for b in num_layers_h:
                        t = str(a) + " " + str(b) + " " + str(c) + " " + str(d) + " " + str(e) + " " + str(f)
                        
                        if t not in already_done:
                            print(a, b, c, d, e, f)
                            model = GraphSAGE(
                                in_channels=dataset.num_node_features, # 300 
                                hidden_channels=a,
                                num_layers=b,
                                out_channels=dataset.num_classes,      # Multi-label output 
                            ).to(device)
    
                            optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
                            criterion = torch.nn.BCEWithLogitsLoss()
    
                            # model training
                            training_vals = []
                            training_epoch_times = []
                            loader = train_loader_main_run
                            model.train()
                            for epoch in range(200):
                                total_loss = 0
                                start_time = time.time()
                                for batch in loader:
                                    batch = batch.to(device)
                                    optimizer.zero_grad()
                                    
                                    # The built-in GraphSAGE takes (x, edge_index)
                                    out = model(batch.x, batch.edge_index)
                                    
                                    # Calculate loss on training nodes within the sampled subgraph [cite: 19]
                                    loss = criterion(out[batch.train_mask], batch.y[batch.train_mask])
                                    
                                    loss.backward()
                                    optimizer.step()
                                    total_loss += loss.item()
                                training_epoch_times.append(time.time() - start_time)
                                epoch_loss = total_loss / len(loader)
                                training_vals.append(epoch_loss)
    
                            peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
                            
                            # model testing
                            testing_vals = []
                            y_true = []
                            y_pred = []
                            
                            loader = test_loader_main_run
                            model.eval()
                            total_loss = 0
                            with torch.no_grad():
                                for batch in loader:
                                    batch = batch.to(device)
                                    optimizer.zero_grad()
                                    
                                    out = model(batch.x, batch.edge_index)
                                    loss = criterion(out[batch.test_mask], batch.y[batch.test_mask])
                                    total_loss += loss.item()
                                    
                                    y_true.append(batch.y[batch.test_mask].cpu())
                                    y_pred.append((out[batch.test_mask] > 0).float().cpu())
                                    
                                epoch_loss = total_loss / len(loader)
                                testing_vals.append(epoch_loss)
    
                            y_true = torch.cat(y_true, dim=0).numpy()
                            y_pred = torch.cat(y_pred, dim=0).numpy()
                            
                            # Accuracy Metric: Micro-F1 (Best for multi-label Yelp)
                            micro_f1 = f1_score(y_true, y_pred, average='micro')
                            macro_f1 = f1_score(y_true, y_pred, average='macro')
                            exact_match = accuracy_score(y_true, y_pred)
                            precision, recall, _, __ = precision_recall_fscore_support(y_true, y_pred, average='micro')
    
                            misc = [peak_vram, micro_f1, macro_f1, exact_match, precision, recall]
    
                            np.save(f'./results/training_vals_{a}_{b}_{c}_{d}_{e}_{f}.npy', np.array(training_vals))
                            np.save(f'./results/training_epoch_times_{a}_{b}_{c}_{d}_{e}_{f}.npy', np.array(training_epoch_times))
                            np.save(f'./results/testing_vals_{a}_{b}_{c}_{d}_{e}_{f}.npy', np.array(testing_vals))
                            np.save(f'./results/pvram_scores_{a}_{b}_{c}_{d}_{e}_{f}.npy', np.array(misc))
                            torch.save(model, f'./results/model_{a}_{b}_{c}_{d}_{e}_{f}.pth')
                            torch.save(optimizer, f'./results/optimizer_{a}_{b}_{c}_{d}_{e}_{f}.pth')

128 3 8000 40 6000 60
256 1 8000 40 6000 60
256 2 8000 40 6000 60
256 3 8000 40 6000 60
512 1 8000 40 6000 60
512 2 8000 40 6000 60
512 3 8000 40 6000 60
128 1 8000 40 6000 80
128 2 8000 40 6000 80
